In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure we can import from local modules
if '.' not in sys.path:
    sys.path.append('.')

from data_loaders.cars_data import load_full_train_and_test
from utils.preprocessing import build_preprocessor, get_feature_types

%matplotlib inline

: 

## 1. Load Data (Cleaning & Mapping)

In [ ]:
train_path = "../data/train.csv"
test_path = "../data/test.csv"
mapping_dir = "../mapping"

print("Loading data...")
X_train, y_train, X_test, test_ids = load_full_train_and_test(
    train_path=train_path,
    test_path=test_path,
    mapping_dir=mapping_dir,
    debug_cleaning=True  # Enable detailed cleaning logs
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape:  {X_test.shape}")

## 2. Inspect Missing Values (Pre-Imputation)

In [ ]:
def plot_missing(df, title):
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    if missing.empty:
        print(f"No missing values in {title}")
        return
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x=missing.values, y=missing.index)
    plt.title(f"Missing Values in {title}")
    plt.xlabel("Count")
    plt.show()
    return missing

print("--- Train Missing ---")
miss_train = plot_missing(X_train, "Train Data")
print(miss_train)

print("\n--- Test Missing ---")
miss_test = plot_missing(X_test, "Test Data")
print(miss_test)

## 3. Inspect Distributions (Plausibility)

In [ ]:
X_train.describe()

In [ ]:
# Check Target Distribution
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.histplot(y_train, kde=True)
plt.title("Price Distribution")

plt.subplot(1, 2, 2)
sns.histplot(np.log1p(y_train), kde=True)
plt.title("Log(Price) Distribution")
plt.show()

## 4. Apply Preprocessing (Imputation & Encoding)
We verify that the pipeline handles all missing values and produces a valid numeric matrix.

In [ ]:
numeric_features, categorical_features = get_feature_types(X_train)
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

preprocessor = build_preprocessor(numeric_features, categorical_features)

# Fit on Train
X_train_processed = preprocessor.fit_transform(X_train)
# Transform Test
X_test_processed = preprocessor.transform(X_test)

print(f"Processed Train Shape: {X_train_processed.shape}")
print(f"Processed Test Shape: {X_test_processed.shape}")

In [ ]:
# Check for NaNs in processed data
def check_validity(X, name):
    if hasattr(X, "toarray"):
        X = X.toarray()
    
    nans = np.isnan(X).sum()
    infs = np.isinf(X).sum()
    print(f"{name}: NaNs={nans}, Infs={infs}")
    if nans > 0:
        print("WARNING: NaNs found in processed data!")

check_validity(X_train_processed, "Train Processed")
check_validity(X_test_processed, "Test Processed")

In [ ]:
# Feature Names
try:
    feature_names = preprocessor.get_feature_names_out()
    print(f"Total Features: {len(feature_names)}")
    print("First 20 features:", feature_names[:20])
except Exception as e:
    print("Could not retrieve feature names:", e)